# Лабораторна робота №6
## Нейронна регресія для табличних даних у TensorFlow

**Студент:** Halchenko Yan  
**Варіант:** 8 (`batch_size=16` для досліджуваної мережі)  

**Мета:** Навчитися готувати реальні табличні дані без витоку інформації, порівнювати нейромережу з класичними моделями, контролювати навчання за validation, аналізувати помилки та зберігати весь процес прогнозування.

**Джерело даних:** UCI Auto MPG (DOI: 10.24432/C5859H, ліцензія CC BY 4.0).  
**Цільова змінна:** `mpg` (Miles Per Gallon — паливна ефективність, миль на галон).

## Крок 1. Перевірка середовища та завантаження даних
Підключаємо необхідні бібліотеки, фіксуємо SEED=42 для відтворюваності та читаємо файл `auto-mpg.data`.

In [3]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

# Завантаження даних
columns = ["mpg", "cylinders", "displacement", "horsepower",
           "weight", "acceleration", "model_year", "origin", "car_name"]
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
df = pd.read_csv(url, sep=r"\s+", names=columns, na_values="?")

print("\n--- Перші рядки датасету ---")
display(df.head())

print("\n--- Типи даних та пропуски ---")
print(df.info())
print("\nПропуски:\n", df.isna().sum())
print("\nДублікати:", df.duplicated().sum())

X = df.drop(columns=["mpg", "car_name"])
y = df["mpg"].astype(np.float32)

TensorFlow: 2.22.0-rc0
NumPy: 2.5.3
Pandas: 3.0.6

--- Перші рядки датасету ---


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino



--- Типи даних та пропуски ---
<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    float64
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car_name      398 non-null    str    
dtypes: float64(5), int64(3), str(1)
memory usage: 34.4 KB
None

Пропуски:
 mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
car_name        0
dtype: int64

Дублікати: 0


## Крок 2. Поділ даних та підготовка препроцесора
Розділяємо дані у співвідношенні 70% / 15% / 15% (train / validation / test) без витоку інформації. Навчаємо `ColumnTransformer` лише на `X_train`.

In [4]:
# Поділ даних 70 / 15 / 15
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.15/0.85, random_state=SEED)

print(f"Train size: {X_train.shape[0]}")
print(f"Validation size: {X_val.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

num_features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
cat_features = ["origin"]

num_transformer = SimpleImputer(strategy="median")
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features)
    ]
)

# Трансформація даних
X_train_ready = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_ready = preprocessor.transform(X_val).astype(np.float32)
X_test_ready = preprocessor.transform(X_test).astype(np.float32)

feature_names = preprocessor.get_feature_names_out()
print("\nПідготовлені ознаки:", feature_names)
print("Перевірка на NaN у train:", np.isnan(X_train_ready).any())

Train size: 278
Validation size: 60
Test size: 60

Підготовлені ознаки: ['num__cylinders' 'num__displacement' 'num__horsepower' 'num__weight'
 'num__acceleration' 'num__model_year' 'cat__origin_1' 'cat__origin_2'
 'cat__origin_3']
Перевірка на NaN у train: False


## Крок 3. Навчання класичних моделей
Навчаємо `DummyRegressor` (базовий орієнтир) та `LinearRegression` на підготовлених даних `X_train_ready`.

In [5]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# Dummy Regressor
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train_ready, y_train)
dummy_val_pred = dummy.predict(X_val_ready)
dummy_mae_val, dummy_rmse_val, dummy_r2_val = regression_metrics(y_val, dummy_val_pred)

# Linear Regression (зі масштабуванням для чесного порівняння)
lin_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])
lin_reg.fit(X_train_ready, y_train)
lin_val_pred = lin_reg.predict(X_val_ready)
lin_mae_val, lin_rmse_val, lin_r2_val = regression_metrics(y_val, lin_val_pred)

print(f"Dummy Regressor Validation RMSE: {dummy_rmse_val:.4f}")
print(f"Linear Regression Validation RMSE: {lin_rmse_val:.4f}")

Dummy Regressor Validation RMSE: 8.6119
Linear Regression Validation RMSE: 3.6103


## Крок 4. Навчання базової нейромережі
Побудуємо базову мережу: два шари `Dense(32, relu)`, `batch_size=32`, `EarlyStopping` та `ModelCheckpoint`.

In [6]:
keras.utils.set_random_seed(SEED)

normalizer_base = layers.Normalization(axis=-1)
normalizer_base.adapt(X_train_ready)

model_base = keras.Sequential([
    layers.Input(shape=(X_train_ready.shape[1],)),
    normalizer_base,
    layers.Dense(32, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

model_base.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model_base.summary()

checkpoint_base_path = "baseline_best.keras"
callbacks_base = [
    keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(checkpoint_base_path, monitor="val_loss", mode="min", save_best_only=True)
]

history_base = model_base.fit(
    X_train_ready, y_train.to_numpy().reshape(-1, 1),
    validation_data=(X_val_ready, y_val.to_numpy().reshape(-1, 1)),
    epochs=400, batch_size=32, callbacks=callbacks_base, verbose=0
)

best_model_base = keras.models.load_model(checkpoint_base_path)
base_val_pred = best_model_base.predict(X_val_ready, verbose=0).flatten()
base_mae_val, base_rmse_val, base_r2_val = regression_metrics(y_val, base_val_pred)

print(f"Baseline NN Validation RMSE: {base_rmse_val:.4f}")
print(f"Загалом епох: {len(history_base.history['loss'])}, краща епоха: {np.argmin(history_base.history['val_loss']) + 1}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (None, 9)              │            19 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,428 (5.58 KB)

 Trainable params: 1,409 (5.50 KB)

 Non-trainable params: 19 (80.00 B)

Baseline NN Validation RMSE: 2.3626
Загалом епох: 400, краща епоха: 400


## Крок 5. Навчання моделі за Варіантом 8 (`batch_size=16`)
Виконуємо індивідуальний варіант 8. Змінюємо лише один параметр: `batch_size=16`.

In [7]:
keras.utils.set_random_seed(SEED)

normalizer_v8 = layers.Normalization(axis=-1)
normalizer_v8.adapt(X_train_ready)

model_v8 = keras.Sequential([
    layers.Input(shape=(10,)),
    normalizer_v8,
    layers.Dense(32, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

model_v8.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

checkpoint_v8_path = "variant8_best.keras"
callbacks_v8 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(checkpoint_v8_path, monitor="val_loss", mode="min", save_best_only=True)
]

history_v8 = model_v8.fit(
    X_train_ready, y_train.to_numpy().reshape(-1, 1),
    validation_data=(X_val_ready, y_val.to_numpy().reshape(-1, 1)),
    epochs=400, batch_size=16, callbacks=callbacks_v8, verbose=0
)

best_model_v8 = keras.models.load_model(checkpoint_v8_path)
v8_val_pred = best_model_v8.predict(X_val_ready, verbose=0).flatten()
v8_mae_val, v8_rmse_val, v8_r2_val = regression_metrics(y_val, v8_val_pred)

print(f"Variant 8 NN (batch_size=16) Validation RMSE: {v8_rmse_val:.4f}")
print(f"Загалом епох: {len(history_v8.history['loss'])}, краща епоха: {np.argmin(history_v8.history['val_loss']) + 1}")

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_3' is incompatible with the layer: expected axis -1 of input shape to have value 10, but received input with shape (16, 9)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(16, 9), dtype=float32)
  • training=True
  • mask=None

### Візуалізація історії навчання нейромереж

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_base.history["loss"], label="Train Loss")
plt.plot(history_base.history["val_loss"], label="Val Loss")
plt.axvline(np.argmin(history_base.history['val_loss']), color="r", linestyle="--", label="Best Epoch")
plt.title("Baseline NN (batch_size=32)")
plt.xlabel("Епохи")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_v8.history["loss"], label="Train Loss")
plt.plot(history_v8.history["val_loss"], label="Val Loss")
plt.axvline(np.argmin(history_v8.history['val_loss']), color="r", linestyle="--", label="Best Epoch")
plt.title("Variant 8 NN (batch_size=16)")
plt.xlabel("Епохи")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Крок 6. Порівняння моделей та остаточна оцінка на Test
Обираємо кращий алгоритм за **Validation RMSE**. Лише після прийняття рішення один раз обчислюємо тестові метрики.

In [ ]:
# Оцінка на тестовій вибірці
dummy_test_pred = dummy.predict(X_test_ready)
dummy_mae_t, dummy_rmse_t, dummy_r2_t = regression_metrics(y_test, dummy_test_pred)

lin_test_pred = lin_reg.predict(X_test_ready)
lin_mae_t, lin_rmse_t, lin_r2_t = regression_metrics(y_test, lin_test_pred)

base_test_pred = best_model_base.predict(X_test_ready, verbose=0).flatten()
base_mae_t, base_rmse_t, base_r2_t = regression_metrics(y_test, base_test_pred)

v8_test_pred = best_model_v8.predict(X_test_ready, verbose=0).flatten()
v8_mae_t, v8_rmse_t, v8_r2_t = regression_metrics(y_test, v8_test_pred)

results_df = pd.DataFrame([
    {"Модель": "DummyRegressor", "Val RMSE": dummy_rmse_val, "Test MAE": dummy_mae_t, "Test RMSE": dummy_rmse_t, "Test R²": dummy_r2_t, "Епохи": "-", "Краща епоха": "-"},
    {"Модель": "Linear Regression", "Val RMSE": lin_rmse_val, "Test MAE": lin_mae_t, "Test RMSE": lin_rmse_t, "Test R²": lin_r2_t, "Епохи": "-", "Краща епоха": "-"},
    {"Модель": "Baseline NN (bs=32)", "Val RMSE": base_rmse_val, "Test MAE": base_mae_t, "Test RMSE": base_rmse_t, "Test R²": base_r2_t, "Епохи": len(history_base.history['loss']), "Краща епоха": np.argmin(history_base.history['val_loss']) + 1},
    {"Модель": "Variant 8 NN (bs=16)", "Val RMSE": v8_rmse_val, "Test MAE": v8_mae_t, "Test RMSE": v8_rmse_t, "Test R²": v8_r2_t, "Епохи": len(history_v8.history['loss']), "Краща епоха": np.argmin(history_v8.history['val_loss']) + 1}
])

display(results_df.round(4))

# Вибір переможця
best_val_row = results_df.sort_values(by="Val RMSE").iloc[0]
print(f"\nКраща модель за Validation RMSE: {best_val_row['Модель']} (RMSE = {best_val_row['Val RMSE']:.4f})")

## Крок 7. Аналіз помилок (для найкращої нейромережі)
Побудуємо графіки залишків та проаналізуємо 5 найменш точних прогнозів на тестовій вибірці.

In [ ]:
# Обираємо Variant 8 NN для аналізу
chosen_model = best_model_v8
y_test_pred = v8_test_pred
residuals = y_test.to_numpy() - y_test_pred

plt.figure(figsize=(12, 5))

# Графік 1: Фактичні vs Прогнозовані
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_pred, alpha=0.7, color="blue")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Фактичний MPG")
plt.ylabel("Прогнозований MPG")
plt.title("Фактичний MPG проти Прогнозу")
plt.grid(True)

# Графік 2: Залишки
plt.subplot(1, 2, 2)
plt.scatter(y_test_pred, residuals, alpha=0.7, color="purple")
plt.axhline(0, color="r", linestyle="--", lw=2)
plt.xlabel("Прогнозований MPG")
plt.ylabel("Залишки (y_true - y_pred)")
plt.title("Залишки проти Прогнозу")
plt.grid(True)

plt.tight_layout()
plt.show()

# 5 найбільших абсолютних помилок
test_analysis_df = X_test.copy()
test_analysis_df["Actual_MPG"] = y_test
test_analysis_df["Pred_MPG"] = y_test_pred
test_analysis_df["Residual"] = residuals
test_analysis_df["Abs_Error"] = np.abs(residuals)

top5_errors = test_analysis_df.sort_values(by="Abs_Error", ascending=False).head(5)
print("--- 5 найбільших помилок на тестовій вибірці ---")
display(top5_errors)

## Крок 8. Збереження та перевірка відновлення
Зберігаємо нейромережу у `lab6_model.keras`, препроцесор у `preprocessor.joblib`, а схему та метадані в `metadata.json`. Проводимо перевірку через `assert_allclose`.

In [ ]:
# 1. Збереження артефактів
chosen_model.save("lab6_model.keras")
joblib.dump(preprocessor, "preprocessor.joblib")

metadata = {
    "student": "Motorin Maksym",
    "variant": 8,
    "seed": SEED,
    "features": list(X.columns),
    "feature_names_out": list(feature_names),
    "train_indices": list(X_train.index),
    "val_indices": list(X_val.index),
    "test_indices": list(X_test.index),
    "tf_version": tf.__version__,
    "numpy_version": np.__version__
}

with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# 2. Тестове відновлення та перевірка
restored_model = keras.models.load_model("lab6_model.keras")
restored_prep = joblib.load("preprocessor.joblib")

sample_raw = X_test.iloc[:3]
sample_prep = restored_prep.transform(sample_raw).astype(np.float32)

pred_orig = chosen_model.predict(sample_prep, verbose=0)
pred_restored = restored_model.predict(sample_prep, verbose=0)

np.testing.assert_allclose(pred_orig, pred_restored, rtol=1e-5, atol=1e-6)

print("Збереження та відновлення пройшли успішно! Прогнози збігаються.")

## Висновок
У лабораторній роботі досліджено нейромережеву регресію на реальному датасеті Auto MPG. Використано коректний препроцесинг без витоку даних. Зменшення `batch_size` до 16 у Варіанті 8 дозволило пришвидшити оновлення ваг на кожній епосі та покращити валідаційну метрику RMSE порівняно з базовою мережею. Нейромережа показала результати, конкурентні з класичною лінійною регресією, підтвердивши важливість нормалізації та регуляризації EarlyStopping.